In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# PySpark Join Optimization Examples 

In [1]:
# Spark setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, lit

spark = SparkSession.builder.appName("JoinExamples").getOrCreate()

# Sample data
df_large = spark.createDataFrame([
    (1, "A"),
    (2, "B"),
    (3, "C"),
    (4, "D")
], ["id", "value"])

df_small = spark.createDataFrame([
    (1, "alpha"),
    (2, "beta")
], ["id", "desc"])


## 1️⃣ Broadcast Join

In [2]:
# Broadcast join avoids shuffle
result = df_large.join(broadcast(df_small), "id")
result.show()

+---+-----+-----+
| id|value| desc|
+---+-----+-----+
|  1|    A|alpha|
|  2|    B| beta|
+---+-----+-----+



## 2️⃣ Sort Merge Join

In [3]:
spark.conf.set("spark.sql.join.preferSortMergeJoin", True)

result = df_large.join(df_small, "id")
result.show()


+---+-----+-----+
| id|value| desc|
+---+-----+-----+
|  1|    A|alpha|
|  2|    B| beta|
+---+-----+-----+



## 3️⃣ Shuffle Hash Join

In [4]:
spark.conf.set("spark.sql.join.preferSortMergeJoin", False)
spark.conf.set("spark.sql.join.forceHashJoin", True)

result = df_large.join(df_small, "id")
result.show()

+---+-----+-----+
| id|value| desc|
+---+-----+-----+
|  1|    A|alpha|
|  2|    B| beta|
+---+-----+-----+



## 4️⃣ Adaptive Query Execution (AQE)

In [5]:
spark.conf.set("spark.sql.adaptive.enabled", True)
spark.conf.set("spark.sql.adaptive.broadcastJoinThreshold", 50)

result = df_large.join(df_small, "id")
result.show()


+---+-----+-----+
| id|value| desc|
+---+-----+-----+
|  1|    A|alpha|
|  2|    B| beta|
+---+-----+-----+



## 5️⃣ Dynamic Partition Pruning

In [6]:
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning", True)

df_fact = spark.createDataFrame([
    (1, "USA"),
    (2, "CAN"),
    (3, "MEX")
], ["country_id", "country"])

df_dim = spark.createDataFrame([
    (1, "North America")
], ["country_id", "region"])

result = df_fact.join(df_dim, "country_id")
result.show()


+----------+-------+-------------+
|country_id|country|       region|
+----------+-------+-------------+
|         1|    USA|North America|
+----------+-------+-------------+



## 6️⃣ Handle Data Skew (Salting)

In [7]:
df_skewed = spark.createDataFrame([
    (1, "A"),
    (1, "B"),
    (1, "C"),
    (2, "D")
], ["id", "value"])

df_dim_salt = spark.createDataFrame([
    (1, "alpha"),
    (2, "beta")
], ["id", "desc"])

df_skewed_salted = df_skewed.withColumn("salt", col("id") % 2)
df_dim_salted = df_dim_salt.withColumn("salt", col("id") % 2)

result = df_skewed_salted.join(df_dim_salted, ["id", "salt"])
result.show()


+---+----+-----+-----+
| id|salt|value| desc|
+---+----+-----+-----+
|  1|   1|    A|alpha|
|  1|   1|    B|alpha|
|  1|   1|    C|alpha|
|  2|   0|    D| beta|
+---+----+-----+-----+



## 7️⃣ Bucketing

In [8]:
df_large.write.bucketBy(4, "id").sortBy("id").saveAsTable("large_bucketed")
df_small.write.bucketBy(4, "id").sortBy("id").saveAsTable("small_bucketed")

df_large_b = spark.table("large_bucketed")
df_small_b = spark.table("small_bucketed")

result = df_large_b.join(df_small_b, "id")
result.show()


+---+-----+-----+
| id|value| desc|
+---+-----+-----+
|  2|    B| beta|
|  1|    A|alpha|
+---+-----+-----+



## 8️⃣ Join Order Optimization

In [9]:
df_big = spark.range(0, 1000).withColumn("flag", lit("X"))
df_filter = spark.createDataFrame([(1,), (2,)], ["id"])

df_big_filtered = df_big.filter(col("id") < 10)

result = df_big_filtered.join(df_filter, "id")
result.show()


+---+----+
| id|flag|
+---+----+
|  1|   X|
|  2|   X|
+---+----+



## 9️⃣ Performance Checklist

In [10]:
spark.conf.set("spark.sql.adaptive.enabled", True)

df_filtered = df_large.filter(col("id") > 1)
df_cached = df_filtered.cache()

result = df_cached.join(broadcast(df_small), "id")
result.show()


+---+-----+----+
| id|value|desc|
+---+-----+----+
|  2|    B|beta|
+---+-----+----+



## 🔟 Common Join Mistakes (Corrected)

In [11]:
# Joining before filtering
correct = df_large.filter(col("id") < 3).join(df_small, "id")
correct.show()

# Broadcasting only small tables
correct2 = df_large.join(broadcast(df_small), "id")
correct2.show()

# Avoid accidental cross joins
correct3 = df_large.join(df_small, "id")
correct3.show()


+---+-----+-----+
| id|value| desc|
+---+-----+-----+
|  1|    A|alpha|
|  2|    B| beta|
+---+-----+-----+

+---+-----+-----+
| id|value| desc|
+---+-----+-----+
|  1|    A|alpha|
|  2|    B| beta|
+---+-----+-----+

+---+-----+-----+
| id|value| desc|
+---+-----+-----+
|  1|    A|alpha|
|  2|    B| beta|
+---+-----+-----+



# Point‑form summaries for all 10 Spark join optimization techniques.


#### 1. Broadcast Join
Sends the small table to all executors

Eliminates shuffle

Best when one table is tiny (e.g., dimensions)

Fastest join type for small‑vs‑large joins

Broadcast small table to all executors

Eliminates shuffle

Fastest join for small‑vs‑large tables

Use when one side < broadcast threshold

#### 2. Sort Merge Join
Default join for large datasets

Requires shuffle + sorting on both sides

Scales well for big distributed joins

Slower than broadcast but stable for large data

#### 3. Shuffle Hash Join
Uses hashing instead of sorting

Faster when one side fits in memory

Reduces sort overhead

Good for medium‑sized tables

#### 4. Adaptive Query Execution (AQE)
Optimizes join strategy at runtime

Can convert Sort Merge → Broadcast automatically

Handles skew by splitting skewed partitions

Adjusts shuffle partitions dynamically

#### 5. Dynamic Partition Pruning
Skips irrelevant partitions during joins

Reduces I/O and scan time

Ideal for star schema fact‑to‑dimension joins

Enabled automatically with AQE

#### 6. Handle Data Skew (Salting)
Adds a "salt" column to distribute skewed keys

Prevents one executor from overloading

Works well when a few keys dominate the dataset

AQE skew optimization can also help

#### 7. Bucketing
Pre‑splits data into fixed buckets based on join key

Minimizes shuffle for repeated joins

Works best when both tables are bucketed on same key

Ideal for ETL pipelines with repeated joins

#### 8. Join Order Optimization
Filter early to reduce data size

Join smaller datasets first

Avoid wide transformations before joins

Reduces intermediate shuffle and memory pressure

#### 9. Performance Checklist
Broadcast small tables

Filter before joining

Cache reused DataFrames

Enable AQE

Avoid unnecessary wide transformations

Monitor Spark UI for skew/shuffle issues

#### 10. Common Join Mistakes
Joining before filtering → causes unnecessary shuffle

Broadcasting large tables → causes memory blowups

Ignoring skew → leads to slow tasks

Missing partition pruning → scans too much data

Accidental cross joins → huge explosion in rows
